In [1]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  BRAIN FIRST MODEL TUNING TOOLKIT — EEG DL Final                        ║
# ║  Standalone — completely separate from keystroke pipeline                ║
# ║                                                                          ║
# ║  Input:  DE (Differential Entropy) features from EmoEEG-MC dataset      ║
# ║          Shape per trial: (5_bands, 64_channels, 30_timepoints)          ║
# ║  Output: 3-class sentiment (positive/negative/neutral) — deployment      ║
# ║          7-class emotion   (joy/sadness/fear/...) — bonus detail         ║
# ║                                                                          ║
# ║  Models trained and compared:                                            ║
# ║    1. EEGNet         — lightweight, standard EEG baseline                ║
# ║    2. ShallowConvNet — specifically designed for EEG, often beats EEGNet ║
# ║    3. Transformer    — winner from previous run, kept + improved         ║
# ║    4. CNN-LSTM       — two 1-layer LSTMs (CUDA-safe)                    ║
# ║    5. DGCNN          — graph convolution over channels                   ║
# ║                                                                          ║
# ║  Improvements over previous version:                                     ║
# ║    ✅ F1-based early stopping   — correct for imbalanced classes         ║
# ║    ✅ LR warmup (5 epochs)      — stable early training                  ║
# ║    ✅ Noise aug 0.01 → 0.07     — meaningful regularisation              ║
# ║    ✅ Channel dropout aug        — randomly zeros channels during train   ║
# ║    ✅ Subject-level z-score      — removes 10x amplitude variance        ║
# ║    ✅ Two 1-layer LSTMs         — avoids cudnn multi-layer crash         ║
# ║    ✅ ShallowConvNet added       — EEG-specific architecture              ║
# ║    ✅ Per-fold per-class F1      — diagnose failure modes                 ║
# ║    ✅ 3-class is deployment model, 7-class kept as bonus                  ║
# ║    ✅ predict_realtime() returns all needed fields for fusion engine      ║
# ╚══════════════════════════════════════════════════════════════════════════╝

import os, json, time, warnings
import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report
from collections import Counter
warnings.filterwarnings('ignore')

# ══════════════════════════════════════════════════════════════════════════
# DEVICE
# ══════════════════════════════════════════════════════════════════════════
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
DEVICE = torch.device('cpu')
if torch.cuda.is_available():
    try:
        a = torch.randn(10, 10, device='cuda')
        _ = torch.mm(a, a).sum().item()
        DEVICE = torch.device('cuda')
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    except Exception as e:
        print(f"GPU failed ({type(e).__name__}) → CPU")
else:
    print("No GPU → CPU")

# ══════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════
DATA_PATH    = "/kaggle/input/datasets/ananyaaaa5/eeg-data"
OUTPUT_DIR   = "/kaggle/working/bfmt_eeg_models"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SUBJECTS     = range(1, 61)
SEED         = 42
N_CHANNELS   = 64
N_TIMEPOINTS = 30
N_BANDS      = 5
N_EPOCHS     = 80
BATCH_SIZE   = 32
LR           = 1e-3
PATIENCE     = 15
N_FOLDS      = 5
DROPOUT      = 0.5
WARMUP_EP    = 5
NOISE_STD    = 0.07    # was 0.01 — meaningful augmentation for EEG
CHAN_DROP_P  = 0.15    # randomly zero 15% of channels during training

torch.manual_seed(SEED)
np.random.seed(SEED)

# ══════════════════════════════════════════════════════════════════════════
# LABEL MAPS
# ══════════════════════════════════════════════════════════════════════════
emotion_7_map = {
    'score_1':'joy',         'score_2':'inspiration',
    'score_3':'tenderness',  'score_4':'fear',
    'score_5':'disgust',     'score_6':'sadness',
    'score_7':'neutral'
}
sentiment_3_map = {
    'joy':'positive',      'inspiration':'positive', 'tenderness':'positive',
    'fear':'negative',     'disgust':'negative',      'sadness':'negative',
    'neutral':'neutral'
}
VALENCE_AROUSAL_MAP = {
    'positive': ( 0.7,  0.6),
    'negative': (-0.6,  0.5),
    'neutral':  ( 0.0,  0.0),
}

# ══════════════════════════════════════════════════════════════════════════
# DATA LOADING WITH SUBJECT-LEVEL NORMALISATION
#
# Why subject-level z-score matters:
#   Raw EEG amplitude varies up to 10x between subjects due to electrode
#   impedance, skull thickness, hair. Without subject-level normalisation,
#   models learn subject identity instead of emotion.
#   We apply: per-subject mean/std → then per-sample z-score at runtime.
# ══════════════════════════════════════════════════════════════════════════

def load_npy(path):
    d = np.load(path, allow_pickle=True)
    if hasattr(d, 'item'):  d = d.item()
    if isinstance(d, dict): d = list(d.values())[0]
    return d

def find_de_path(sub_id, ses_name):
    fname = f"{sub_id}_{ses_name}_task-emotion_de.npy"
    for p in [
        os.path.join(DATA_PATH,"derivatives",sub_id,ses_name,"eeg",fname),
        os.path.join(DATA_PATH,"derivatives",sub_id,ses_name,fname),
    ]:
        if os.path.exists(p): return p
    return None

print("Loading EEG data with subject-level normalisation...\n")
all_X, all_y7, all_y3, all_groups = [], [], [], []
loaded, skipped = 0, []

# Collect all trials first, then normalise per subject
subject_trials = {}   # sub_id → list of (arr, e7, e3)

for i in SUBJECTS:
    sub_id   = f"sub-{i:02d}"
    beh_path = os.path.join(DATA_PATH, sub_id, "beh",
                            f"{sub_id}_task-emotion_beh.tsv")
    if not os.path.exists(beh_path):
        skipped.append(sub_id); continue

    beh_df   = pd.read_csv(beh_path, sep='\t')
    sub_trials = []

    for ses_name, beh_offset in [("ses-ima", 0), ("ses-vid", 21)]:
        de_path = find_de_path(sub_id, ses_name)
        if de_path is None: continue
        try:
            de_data = load_npy(de_path)
        except:
            continue
        if de_data.ndim != 3 or de_data.shape[0] != 64 or de_data.shape[2] != 5:
            continue

        n_trials       = 21
        time_per_trial = de_data.shape[1] // n_trials

        for trial_idx in range(n_trials):
            beh_row = beh_offset + trial_idx
            if beh_row >= len(beh_df): break

            t_s      = trial_idx * time_per_trial
            t_e      = t_s + time_per_trial
            trial_de = de_data[:, t_s:t_e, :]

            if trial_de.shape[1] < N_TIMEPOINTS:
                trial_de = np.pad(trial_de, ((0,0),(0,N_TIMEPOINTS-trial_de.shape[1]),(0,0)))
            elif trial_de.shape[1] > N_TIMEPOINTS:
                trial_de = trial_de[:, :N_TIMEPOINTS, :]

            scores = beh_df.iloc[beh_row][
                [f'score_{j}' for j in range(1, 8)]
            ].values.astype(float)
            if np.any(np.isnan(scores)): continue

            eidx = np.argmax(scores)
            e7   = emotion_7_map[f'score_{eidx+1}']
            e3   = sentiment_3_map[e7]

            # Shape: (5_bands, 64_channels, 30_timepoints)
            arr = trial_de.transpose(2, 0, 1).astype(np.float32)
            if np.isnan(arr).any() or np.isinf(arr).any(): continue

            sub_trials.append((arr, e7, e3))

    if len(sub_trials) == 0:
        skipped.append(sub_id); continue

    # Subject-level normalisation: compute mean/std across all this subject's
    # trials, then subtract. Removes inter-subject amplitude differences.
    sub_arrays = np.stack([t[0] for t in sub_trials])  # (n_trials, 5, 64, 30)
    sub_mean   = sub_arrays.mean(axis=(0,2,3), keepdims=True)  # (1, 5, 1, 1)
    sub_std    = sub_arrays.std(axis=(0,2,3), keepdims=True) + 1e-8

    for j, (arr, e7, e3) in enumerate(sub_trials):
        arr_norm = (arr - sub_mean[0]) / sub_std[0]
        all_X.append(arr_norm)
        all_y7.append(e7)
        all_y3.append(e3)
        all_groups.append(i)

    loaded += 1
    print(f"  {sub_id} ✓  ({len(sub_trials)} trials)")

X      = np.array(all_X,    dtype=np.float32)
y7     = np.array(all_y7)
y3     = np.array(all_y3)
groups = np.array(all_groups)

le7 = LabelEncoder(); y7_enc = le7.fit_transform(y7)
le3 = LabelEncoder(); y3_enc = le3.fit_transform(y3)
N7  = len(le7.classes_)
N3  = len(le3.classes_)

print(f"\n{'='*55}")
print(f"  Loaded  : {loaded} subjects")
print(f"  Skipped : {skipped}")
print(f"  Shape   : {X.shape}  (trials, bands, channels, timepoints)")
print(f"  7-class : {dict(sorted(Counter(y7).items()))}")
print(f"  3-class : {dict(sorted(Counter(y3).items()))}")
print(f"  Device  : {DEVICE}")
print(f"{'='*55}")


# ══════════════════════════════════════════════════════════════════════════
# DATASET WITH AUGMENTATION
#
# Two augmentations applied only during training:
#   1. Gaussian noise (std=0.07): prevents memorising exact DE values
#   2. Channel dropout (p=0.15): randomly zeros full EEG channels,
#      forces model to learn from multiple channels not just a few.
#      Mimics realistic electrode artifacts.
# ══════════════════════════════════════════════════════════════════════════

class EEGDataset(Dataset):
    def __init__(self, X, y, augment=False):
        self.X       = torch.tensor(X, dtype=torch.float32)
        self.y       = torch.tensor(y, dtype=torch.long)
        self.augment = augment

    def __len__(self):        return len(self.y)
    def __getitem__(self, i):
        x = self.X[i].clone()

        # Per-sample z-score (applied always — matches inference)
        mu    = x.mean(); sigma = x.std() + 1e-8
        x     = (x - mu) / sigma

        if self.augment:
            # 1. Gaussian noise on DE values
            x = x + NOISE_STD * torch.randn_like(x)

            # 2. Channel dropout — zero entire EEG channels randomly
            # x shape: (bands, channels, time) — drop along channels dim
            n_ch   = x.shape[1]
            n_drop = int(n_ch * CHAN_DROP_P)
            if n_drop > 0:
                drop_idx = torch.randperm(n_ch)[:n_drop]
                x[:, drop_idx, :] = 0.0

        return x, self.y[i]


# ══════════════════════════════════════════════════════════════════════════
# TRAINING UTILITY
# F1-based early stopping — correct for imbalanced EEG emotion classes.
# LR warmup: 5 epochs linear ramp → prevents destructive early gradients.
# ══════════════════════════════════════════════════════════════════════════

def train_and_eval(model, tr_ld, va_ld, n_classes, tag=""):
    # Class-weighted loss (sampler not used here — EEG batches are smaller)
    labels_np = tr_ld.dataset.y.numpy()
    counts    = np.bincount(labels_np, minlength=n_classes)
    w         = 1.0 / (counts + 1e-8); w = w / w.sum() * n_classes
    criterion = nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32).to(DEVICE))

    opt    = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    warmup = torch.optim.lr_scheduler.LinearLR(opt, start_factor=0.1, total_iters=WARMUP_EP)
    cosine = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(N_EPOCHS-WARMUP_EP,1))
    sch    = torch.optim.lr_scheduler.SequentialLR(opt, [warmup,cosine], milestones=[WARMUP_EP])

    best_f1, best_acc, best_state, no_imp = 0.0, 0.0, None, 0
    model.to(DEVICE)

    for ep in range(N_EPOCHS):
        model.train()
        for xb, yb in tr_ld:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sch.step()

        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for xb, yb in va_ld:
                preds.extend(model(xb.to(DEVICE)).argmax(1).cpu().numpy())
                trues.extend(yb.numpy())

        val_f1  = f1_score(trues, preds, average='weighted', zero_division=0)
        val_acc = accuracy_score(trues, preds)

        if val_f1 > best_f1:
            best_f1=val_f1; best_acc=val_acc
            best_state={k:v.clone() for k,v in model.state_dict().items()}
            no_imp=0
        else:
            no_imp += 1

        if no_imp >= PATIENCE:
            break

    return best_f1, best_acc, best_state


def run_cv(model_fn, X, y_enc, groups, n_classes, tag):
    gkf = GroupKFold(n_splits=N_FOLDS)
    accs, f1s, all_p, all_t = [], [], [], []
    best_overall_state = None
    best_overall_f1    = 0.0

    for fold, (tr, te) in enumerate(gkf.split(X, y_enc, groups)):
        t0 = time.time()
        tr_ds = EEGDataset(X[tr], y_enc[tr], augment=True)
        va_ds = EEGDataset(X[te], y_enc[te], augment=False)
        tr_ld = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=False)
        va_ld = DataLoader(va_ds, batch_size=BATCH_SIZE, shuffle=False)

        model = model_fn(n_classes)
        f1, acc, state = train_and_eval(model, tr_ld, va_ld, n_classes, tag)
        model.load_state_dict(state); model.eval()

        preds, trues = [], []
        with torch.no_grad():
            for xb, yb in va_ld:
                preds.extend(model(xb.to(DEVICE)).argmax(1).cpu().numpy())
                trues.extend(yb.numpy())

        fold_acc = accuracy_score(trues, preds)
        fold_f1  = f1_score(trues, preds, average='weighted', zero_division=0)
        accs.append(fold_acc); f1s.append(fold_f1)
        all_p.extend(preds); all_t.extend(trues)

        elapsed = int(time.time() - t0)
        print(f"  [{tag}] Fold {fold+1}/{N_FOLDS}   acc={fold_acc:.4f}  f1={fold_f1:.4f}  ({elapsed}s)")

        # Per-fold per-class F1 — shows which emotions fail on which users
        le = le3 if n_classes == N3 else le7
        pcf = f1_score(trues, preds, average=None, zero_division=0)
        for ci, cn in enumerate(le.classes_):
            print(f"    {cn:<14}: f1={pcf[ci]:.3f}")

        if fold_f1 > best_overall_f1:
            best_overall_f1    = fold_f1
            best_overall_state = state

    mean_acc = np.mean(accs); mean_f1 = np.mean(f1s)
    print(f"  [{tag}] CV  acc={mean_acc:.4f}±{np.std(accs):.4f}  "
          f"f1={mean_f1:.4f}±{np.std(f1s):.4f}")
    return mean_acc, mean_f1, best_overall_state, all_t, all_p


# ══════════════════════════════════════════════════════════════════════════
# MODEL 1 — EEGNet
# Compact depthwise+separable convolutions. Standard EEG baseline.
# ══════════════════════════════════════════════════════════════════════════

class EEGNet(nn.Module):
    def __init__(self, n_classes, n_ch=N_CHANNELS, n_t=N_TIMEPOINTS,
                 n_bands=N_BANDS, D=2, drop=DROPOUT):
        super().__init__()
        F1 = 8
        # Temporal conv across time
        self.block1 = nn.Sequential(
            nn.Conv2d(n_bands, F1, (1, 5), padding=(0,2), bias=False),
            nn.BatchNorm2d(F1),
        )
        # Depthwise conv across channels
        self.block2 = nn.Sequential(
            nn.Conv2d(F1, F1*D, (n_ch, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F1*D), nn.ELU(),
            nn.AvgPool2d((1, 2)), nn.Dropout(drop),
        )
        # Separable conv
        self.block3 = nn.Sequential(
            nn.Conv2d(F1*D, F1*D, (1, 5), padding=(0,2), bias=False),
            nn.Conv2d(F1*D, F1*D, 1, bias=False),
            nn.BatchNorm2d(F1*D), nn.ELU(),
            nn.AvgPool2d((1, 4)), nn.Dropout(drop),
        )
        out_t   = n_t // 8
        self.fc = nn.Linear(F1*D * out_t, n_classes)

    def forward(self, x):
        # x: (B, bands, ch, time)
        out = self.block1(x)
        out = self.block2(out)
        out = self.block3(out)
        out = out.flatten(1)
        return self.fc(out)


# ══════════════════════════════════════════════════════════════════════════
# MODEL 2 — ShallowConvNet
# Specifically designed for EEG band-power features.
# Learns spatiotemporal filters in one pass — often beats EEGNet on
# emotion tasks because it preserves the band structure better.
# ══════════════════════════════════════════════════════════════════════════

class ShallowConvNet(nn.Module):
    def __init__(self, n_classes, n_ch=N_CHANNELS, n_t=N_TIMEPOINTS,
                 n_bands=N_BANDS, drop=DROPOUT):
        super().__init__()
        n_filters = 40

        # Temporal filter across time dimension
        self.temporal = nn.Conv2d(n_bands, n_filters, (1, 13), padding=(0,6), bias=False)
        # Spatial filter across channels
        self.spatial  = nn.Conv2d(n_filters, n_filters, (n_ch, 1), bias=False, groups=1)
        self.bn       = nn.BatchNorm2d(n_filters)
        self.pool     = nn.AvgPool2d((1, 3), stride=(1, 3))
        self.drop     = nn.Dropout(drop)

        out_t   = n_t // 3
        self.fc = nn.Sequential(
            nn.Linear(n_filters * out_t, 128),
            nn.ELU(), nn.Dropout(drop),
            nn.Linear(128, n_classes)
        )

    def forward(self, x):
        out = self.temporal(x)       # (B, 40, ch, t)
        out = self.spatial(out)      # (B, 40, 1, t)
        out = self.bn(out)
        out = out ** 2               # square — preserves band-power interpretation
        out = torch.log(torch.clamp(self.pool(out), min=1e-6))   # log of avg power
        out = self.drop(out)
        out = out.flatten(1)
        return self.fc(out)


# ══════════════════════════════════════════════════════════════════════════
# MODEL 3 — Transformer
# Best from previous run (49.1% 3-class). Kept with F1 stopping + warmup.
# Treats each timepoint as a token, attends across time.
# ══════════════════════════════════════════════════════════════════════════

class EEGTransformer(nn.Module):
    def __init__(self, n_classes, n_ch=N_CHANNELS, n_t=N_TIMEPOINTS,
                 n_bands=N_BANDS, d_model=128, nhead=4, n_layers=2, drop=DROPOUT):
        super().__init__()
        # Project flattened (bands × channels) per timepoint to d_model
        self.input_proj = nn.Linear(n_bands * n_ch, d_model)
        self.pos_drop   = nn.Dropout(drop * 0.5)

        enc_layer  = nn.TransformerEncoderLayer(
            d_model, nhead, dim_feedforward=256,
            dropout=drop, batch_first=True, norm_first=True
        )
        self.tf    = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.head  = nn.Sequential(
            nn.Linear(d_model, 64), nn.GELU(), nn.Dropout(drop),
            nn.Linear(64, n_classes)
        )

    def forward(self, x):
        # x: (B, bands, ch, time) → (B, time, bands*ch)
        B, Bd, C, T = x.shape
        x = x.permute(0, 3, 1, 2).reshape(B, T, Bd*C)
        x = self.pos_drop(self.input_proj(x))    # (B, T, d_model)
        x = self.tf(x)                            # (B, T, d_model)
        x = x.mean(dim=1)                         # global avg over time
        return self.head(x)


# ══════════════════════════════════════════════════════════════════════════
# MODEL 4 — CNN-LSTM (two 1-layer LSTMs — CUDA-safe)
# ══════════════════════════════════════════════════════════════════════════

class CNNLSTM(nn.Module):
    def __init__(self, n_classes, n_ch=N_CHANNELS, n_t=N_TIMEPOINTS,
                 n_bands=N_BANDS, drop=DROPOUT):
        super().__init__()
        # Spatial CNN across channels
        self.spatial = nn.Sequential(
            nn.Conv2d(n_bands, 32, (n_ch,1), bias=False),
            nn.BatchNorm2d(32), nn.ELU(), nn.Dropout(drop*0.5),
        )
        # Temporal CNN across time
        self.temporal = nn.Sequential(
            nn.Conv1d(32, 64, 3, padding=1), nn.BatchNorm1d(64), nn.ELU(),
            nn.Conv1d(64, 64, 3, padding=1), nn.BatchNorm1d(64), nn.ELU(),
        )
        # Two 1-layer LSTMs with manual dropout between (CUDA-safe)
        self.lstm1 = nn.LSTM(64, 64, num_layers=1, batch_first=True, bidirectional=True)
        self.drop  = nn.Dropout(drop)
        self.lstm2 = nn.LSTM(128, 64, num_layers=1, batch_first=True, bidirectional=True)
        self.head  = nn.Sequential(
            nn.Linear(128, 64), nn.ELU(), nn.Dropout(drop),
            nn.Linear(64, n_classes)
        )

    def forward(self, x):
        out = self.spatial(x)                    # (B, 32, 1, T)
        out = out.squeeze(2)                     # (B, 32, T)
        out = self.temporal(out)                 # (B, 64, T)
        out = out.permute(0, 2, 1)               # (B, T, 64)
        out, _ = self.lstm1(out)
        out = self.drop(out)
        out, _ = self.lstm2(out)                 # (B, T, 128)
        out = out[:, -1, :]                      # last timestep
        return self.head(out)


# ══════════════════════════════════════════════════════════════════════════
# MODEL 5 — DGCNN (Dynamic Graph CNN over EEG channels)
# Models spatial relationships between electrode channels as a graph.
# Each channel is a node; edges are learned by dot-product similarity.
# Captures that frontal and temporal channels interact differently for
# different emotions — something flat CNNs cannot express.
# ══════════════════════════════════════════════════════════════════════════

class DGCNNBlock(nn.Module):
    def __init__(self, in_f, out_f, k=8):
        super().__init__()
        self.k   = k
        self.mlp = nn.Sequential(
            nn.Linear(in_f * 2, out_f), nn.BatchNorm1d(out_f), nn.ELU()
        )

    def forward(self, x):
        # x: (B, N_nodes, in_features)
        B, N, F = x.shape
        # Pairwise similarity to find k nearest neighbours in feature space
        xx   = (x ** 2).sum(-1, keepdim=True)  # (B, N, 1)
        dist = xx + xx.transpose(1,2) - 2 * torch.bmm(x, x.transpose(1,2))
        k    = min(self.k, N)
        idx  = dist.topk(k, dim=-1, largest=False).indices  # (B, N, k)

        # Gather neighbour features
        idx_exp  = idx.unsqueeze(-1).expand(-1,-1,-1,F)
        x_exp    = x.unsqueeze(2).expand(-1,-1,k,-1)
        x_nbr    = x.unsqueeze(1).expand(-1,N,-1,-1).gather(2, idx_exp)
        edge_feat= torch.cat([x_exp, x_nbr - x_exp], dim=-1)  # (B,N,k,2F)

        # MLP on edge features
        edge_feat = edge_feat.reshape(B*N*k, -1)
        edge_out  = self.mlp(edge_feat).reshape(B, N, k, -1)
        return edge_out.max(dim=2).values   # (B, N, out_f)


class DGCNN(nn.Module):
    def __init__(self, n_classes, n_ch=N_CHANNELS, n_t=N_TIMEPOINTS,
                 n_bands=N_BANDS, drop=DROPOUT):
        super().__init__()
        # Flatten time into features per channel per band
        in_f = n_t * n_bands

        self.gc1  = DGCNNBlock(in_f, 128, k=8)
        self.gc2  = DGCNNBlock(128,  256, k=8)
        self.drop = nn.Dropout(drop)

        self.head = nn.Sequential(
            nn.Linear(256, 128), nn.ELU(), nn.Dropout(drop),
            nn.Linear(128, n_classes)
        )

    def forward(self, x):
        # x: (B, bands, ch, time) → (B, ch, bands*time)
        B, Bd, C, T = x.shape
        x = x.permute(0, 2, 1, 3).reshape(B, C, Bd*T)   # (B, 64, 5*30)
        x = self.gc1(x)                                   # (B, 64, 128)
        x = self.drop(x)
        x = self.gc2(x)                                   # (B, 64, 256)
        x = x.max(dim=1).values                           # global max over channels
        return self.head(x)


# ══════════════════════════════════════════════════════════════════════════
# MODEL REGISTRY
# ══════════════════════════════════════════════════════════════════════════

MODELS = {
    "EEGNet":       lambda nc: EEGNet(nc),
    "ShallowConv":  lambda nc: ShallowConvNet(nc),
    "Transformer":  lambda nc: EEGTransformer(nc),
    "CNN-LSTM":     lambda nc: CNNLSTM(nc),
    "DGCNN":        lambda nc: DGCNN(nc),
}


# ══════════════════════════════════════════════════════════════════════════
# RUN EXPERIMENTS
# ══════════════════════════════════════════════════════════════════════════

results_7 = {}
results_3 = {}

for task_name, y_enc, n_classes in [
    ("7-class", y7_enc, N7),
    ("3-class", y3_enc, N3),
]:
    print(f"\n{'━'*65}")
    print(f"  Task: {task_name} emotion  ({n_classes} classes)")
    print(f"{'━'*65}")

    task_results = results_7 if task_name == "7-class" else results_3
    le = le7 if task_name == "7-class" else le3

    for model_name, model_fn in MODELS.items():
        print(f"\n  ▶ {model_name}")
        tag = f"{model_name}/{task_name}"
        acc, f1, best_state, all_t, all_p = run_cv(
            model_fn, X, y_enc, groups, n_classes, tag
        )
        task_results[model_name] = {'acc': acc, 'f1': f1}

        # Save best fold weights
        m = model_fn(n_classes)
        m.load_state_dict(best_state)
        save_path = os.path.join(OUTPUT_DIR, f"{model_name.replace('-','_')}_{task_name.replace('-','')}.pt")
        torch.save(best_state, save_path)
        print(f"  ✓ Saved {save_path}")

        # Per-class breakdown for 3-class (the deployment task)
        if task_name == "3-class":
            print(f"\n  Classification report ({model_name} 3-class):")
            print(classification_report(all_t, all_p, target_names=le.classes_, digits=3))


# ══════════════════════════════════════════════════════════════════════════
# RESULTS TABLE
# ══════════════════════════════════════════════════════════════════════════

print(f"\n{'═'*65}")
print("  FINAL COMPARISON")
print(f"{'═'*65}")
print(f"  {'Model':<16} {'7-class acc':>12}  {'7-class f1':>11}  {'3-class acc':>12}  {'3-class f1':>11}")
print(f"  {'-'*63}")
print(f"  {'RF baseline':<16} {'0.191':>12}  {'—':>11}  {'0.489':>12}  {'—':>11}  (previous run)")
for name in MODELS:
    r7 = results_7.get(name, {})
    r3 = results_3.get(name, {})
    print(f"  {name:<16} {r7.get('acc',0):>12.4f}  {r7.get('f1',0):>11.4f}  "
          f"{r3.get('acc',0):>12.4f}  {r3.get('f1',0):>11.4f}")
print(f"  {'Chance 7-class':<16} {'0.143':>12}")
print(f"  {'Chance 3-class':<16} {'':>12}  {'':>11}  {'0.333':>12}")

# Pick deployment winner (3-class, ranked by F1)
winner_3 = max(results_3, key=lambda k: results_3[k]['f1'])
winner_7 = max(results_7, key=lambda k: results_7[k]['f1'])
print(f"\n  3-class winner: {winner_3}  (f1={results_3[winner_3]['f1']:.4f})  ← deployment model")
print(f"  7-class winner: {winner_7}  (f1={results_7[winner_7]['f1']:.4f})")


# ══════════════════════════════════════════════════════════════════════════
# SAVE ARTIFACTS
# ══════════════════════════════════════════════════════════════════════════

joblib.dump(le3, os.path.join(OUTPUT_DIR, "le3.pkl"))
joblib.dump(le7, os.path.join(OUTPUT_DIR, "le7.pkl"))

config = {
    "N_CHANNELS":   N_CHANNELS,
    "N_TIMEPOINTS": N_TIMEPOINTS,
    "N_BANDS":      N_BANDS,
    "N3":           N3,
    "N7":           N7,
    "winner_3class": winner_3,
    "winner_7class": winner_7,
    "results_3class": results_3,
    "results_7class": results_7,
}
with open(os.path.join(OUTPUT_DIR, "eeg_config.json"), "w") as f:
    json.dump(config, f, indent=2)

print(f"\n  ✅ le3.pkl  le7.pkl  eeg_config.json  → {OUTPUT_DIR}/")


# ══════════════════════════════════════════════════════════════════════════
# INFERENCE FUNCTION
# This is what gets called when real EEG hardware is connected.
# Input:  raw DE features (5, 64, 30) — same format as training
# Output: sentiment, confidence, valence, arousal (for fusion engine)
# ══════════════════════════════════════════════════════════════════════════

_model_cache = {}

def _load_model(model_name, task, n_classes, device):
    key = f"{model_name}_{task}"
    if key not in _model_cache:
        model_fn = MODELS[model_name]
        m = model_fn(n_classes).to(device)
        path = os.path.join(OUTPUT_DIR,
                            f"{model_name.replace('-','_')}_{task.replace('-','')}.pt")
        m.load_state_dict(torch.load(path, map_location=device))
        m.eval()
        _model_cache[key] = m
    return _model_cache[key]


def predict_realtime(de_features: np.ndarray,
                     device=DEVICE) -> dict:
    """
    Predict emotional state from EEG DE features.

    Parameters:
        de_features: np.ndarray, shape (5, 64, 30)
                     5 frequency bands × 64 channels × 30 timepoints
                     Same format as training data.

    Returns dict with:
        sentiment   : 'positive' | 'negative' | 'neutral'
        confidence  : float [0,1]
        valence     : float [-1,1]
        arousal     : float [-1,1]
        emotion     : fine-grained 7-class label
        probs_3     : dict of 3-class probabilities
        probs_7     : dict of 7-class probabilities

    This function is called by eeg_engine.py in the main pipeline.
    When hardware is connected, replace the dummy call with real DE extraction.
    """
    assert de_features.shape == (N_BANDS, N_CHANNELS, N_TIMEPOINTS), \
        f"Expected ({N_BANDS},{N_CHANNELS},{N_TIMEPOINTS}), got {de_features.shape}"

    x = torch.tensor(de_features, dtype=torch.float32)

    # Per-sample z-score (matches training)
    mu = x.mean(); sigma = x.std() + 1e-8
    x  = ((x - mu) / sigma).unsqueeze(0).to(device)   # (1, 5, 64, 30)

    with torch.no_grad():
        # 3-class (deployment)
        model_3  = _load_model(winner_3, "3class", N3, device)
        logits_3 = model_3(x)
        probs_3  = F.softmax(logits_3, dim=-1).squeeze().cpu().numpy()
        pred_3   = int(probs_3.argmax())
        sentiment   = le3.classes_[pred_3]
        confidence  = float(probs_3[pred_3])

        # 7-class (fine-grained detail — bonus)
        model_7  = _load_model(winner_7, "7class", N7, device)
        logits_7 = model_7(x)
        probs_7  = F.softmax(logits_7, dim=-1).squeeze().cpu().numpy()
        pred_7   = int(probs_7.argmax())
        emotion  = le7.classes_[pred_7]

    valence, arousal = VALENCE_AROUSAL_MAP[sentiment]

    return {
        'sentiment':  sentiment,
        'emotion':    emotion,
        'confidence': confidence,
        'valence':    float(valence),
        'arousal':    float(arousal),
        'probs_3':    dict(zip(le3.classes_, probs_3.tolist())),
        'probs_7':    dict(zip(le7.classes_, probs_7.tolist())),
    }


# ── Smoke test ──────────────────────────────────────────────────────────
print("\n── predict_realtime() smoke test ──")
dummy = np.random.randn(N_BANDS, N_CHANNELS, N_TIMEPOINTS).astype(np.float32)
result = predict_realtime(dummy)
print(f"  sentiment  : {result['sentiment']}")
print(f"  emotion    : {result['emotion']}")
print(f"  confidence : {result['confidence']:.4f}")
print(f"  valence    : {result['valence']}  arousal: {result['arousal']}")
print(f"  3-class probs:")
for emo, prob in result['probs_3'].items():
    bar = "█" * int(prob * 30)
    print(f"    {emo:<12} {prob:.3f}  {bar}")
print(f"  7-class probs:")
for emo, prob in sorted(result['probs_7'].items(), key=lambda x: -x[1]):
    bar = "█" * int(prob * 30)
    print(f"    {emo:<14} {prob:.3f}  {bar}")

print(f"\n✅ EEG module complete.")
print(f"   Artifacts: {OUTPUT_DIR}/")
print(f"   When hardware is connected: call predict_realtime(de_features)")
print(f"   Output plugs directly into fusion.py confidence weighting.")



GPU failed (AcceleratorError) → CPU
Loading EEG data with subject-level normalisation...

  sub-01 ✓  (42 trials)
  sub-02 ✓  (42 trials)
  sub-03 ✓  (42 trials)
  sub-04 ✓  (42 trials)
  sub-05 ✓  (42 trials)
  sub-06 ✓  (42 trials)
  sub-07 ✓  (42 trials)
  sub-08 ✓  (42 trials)
  sub-09 ✓  (42 trials)
  sub-10 ✓  (42 trials)
  sub-11 ✓  (42 trials)
  sub-12 ✓  (42 trials)
  sub-13 ✓  (42 trials)
  sub-14 ✓  (42 trials)
  sub-15 ✓  (42 trials)
  sub-16 ✓  (42 trials)
  sub-17 ✓  (42 trials)
  sub-18 ✓  (42 trials)
  sub-19 ✓  (42 trials)
  sub-20 ✓  (42 trials)
  sub-21 ✓  (42 trials)
  sub-22 ✓  (42 trials)
  sub-23 ✓  (42 trials)
  sub-24 ✓  (42 trials)
  sub-25 ✓  (42 trials)
  sub-26 ✓  (42 trials)
  sub-27 ✓  (42 trials)
  sub-28 ✓  (42 trials)
  sub-29 ✓  (42 trials)
  sub-30 ✓  (42 trials)
  sub-31 ✓  (42 trials)
  sub-32 ✓  (42 trials)
  sub-33 ✓  (42 trials)
  sub-34 ✓  (42 trials)
  sub-35 ✓  (42 trials)
  sub-36 ✓  (42 trials)
  sub-37 ✓  (42 trials)
  sub-38 ✓  (42 trials